# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 115879  # Reemplazar por su semilla primigenia

# Cantidad de arbolitos por ensemble (fijado en 32 para la Tarea 03)
PARAM$num_trees_max <- 32

# Espacio de búsqueda / Grid de hiperparámetros a iterar
# NOTA DE TIEMPO (Google Colab):
# Cada corrida de 32 árboles toma aprox. 30 minutos.
# Con 16 combinaciones (2 x 2 x 2 x 2 x 1), el tiempo total estimado es ~8 horas,
# dejando un margen de seguridad amplio respecto al límite máximo de 12 horas de Colab.

# Valores del experimento 420_01
# PARAM$grid <- list(
#   feature_fraction = c(0.3, 0.5, 0.7),
#   maxdepth = c(6, 10),
#   minsplit = c(500, 200),
#   minbucket_fraction = c(0.2, 0.4),
#   cp = c(-1)
# )

PARAM$grid <- list(
  feature_fraction = c(0.70),
  maxdepth = c(12, 14, 16),
  minsplit = c(100, 600, 800),
  minbucket_fraction = c(0.2, 0.3),
  cp = c(-1)
)

In [ ]:
# carpeta de trabajo en Google Drive
setwd("/content/buckets/b1/exp")
experimento <- "exp420_02"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [ ]:
# Bucle anidado para iterar sobre todos los hiperparámetros
iter <- 0

# Si ya existe un resumen previo en Google Drive, lo cargamos para retomar
if (file.exists("resumen_corridas.txt")) {
  tb_resumen_corridas <- fread("resumen_corridas.txt")
} else {
  tb_resumen_corridas <- data.table(
    iter = integer(),
    feature_fraction = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket_fraction = numeric(),
    minbucket = integer(),
    cp = numeric(),
    num_trees = integer(),
    archivo = character(),
    envios_positivos = integer()
  )
}

for (v_ff in PARAM$grid$feature_fraction) {
  for (v_maxdepth in PARAM$grid$maxdepth) {
    for (v_minsplit in PARAM$grid$minsplit) {
      for (v_minbucket_frac in PARAM$grid$minbucket_fraction) {

        # Calcular minbucket como fracción de minsplit
        v_minbucket <- max(1L, as.integer(round(v_minsplit * v_minbucket_frac)))

        for (v_cp in PARAM$grid$cp) {
          iter <- iter + 1

          # Nombre descriptivo del archivo con los parámetros utilizados (minbucket calculado)
          archivo_prediccion <- sprintf(
            "KA420_ff_%.2f_md_%d_ms_%d_mb_%d_cp_%.1f_32trees.csv",
            v_ff, v_maxdepth, v_minsplit, v_minbucket, v_cp
          )

          cat(sprintf("\n[Iteración %d] ff: %.2f | maxdepth: %d | minsplit: %d | minbucket: %d (frac: %.2f) | cp: %.2f\n",
                      iter, v_ff, v_maxdepth, v_minsplit, v_minbucket, v_minbucket_frac, v_cp))

          # Checkpoint: Si el archivo ya existe en Google Drive, saltear
          if (file.exists(archivo_prediccion)) {
            cat(sprintf("  -> El archivo %s ya existe. Salteando...\n", archivo_prediccion))
            next
          }

          # Inicializar acumulador de probabilidades para dfuture
          tb_prediccion <- dfuture[, list(numero_de_cliente)]
          tb_prediccion[, prob_acumulada := 0]

          # Inicializar semilla para reproducibilidad en cada combinación
          set.seed(PARAM$semilla_primigenia)

          param_rpart <- list(
            cp = v_cp,
            maxdepth = v_maxdepth,
            minsplit = v_minsplit,
            minbucket = v_minbucket
          )

          # Generar los 32 arbolitos del ensemble
          for (arbolito in seq(PARAM$num_trees_max)) {
            qty_campos_a_utilizar <- as.integer(length(campos_buenos) * v_ff)

            campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
            campos_random <- paste(campos_random, collapse = " + ")
            formulita <- paste0("clase_ternaria ~ ", campos_random)

            modelo <- rpart(formulita,
              data = dtrain,
              xval = 0,
              control = param_rpart
            )

            prediccion <- predict(modelo, dfuture, type = "prob")
            tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]
          }

          # Calcular corte para los 32 árboles (umbral 1/40 acumulado)
          umbral_corte <- (1 / 40) * PARAM$num_trees_max
          tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

          # Guardar archivo de predicción en Google Drive
          fwrite(
            tb_prediccion[, list(numero_de_cliente, Predicted)],
            file = archivo_prediccion,
            sep = ","
          )

          cant_positivos <- tb_prediccion[, sum(Predicted)]
          cat(sprintf("  -> Guardado: %s (Envíos positivos: %d)\n", archivo_prediccion, cant_positivos))

          # Subida a Kaggle
          comando <- "kaggle competitions submit"
          competencia <- "-c utn-2026-inicial"
          arch <- paste("-f", archivo_prediccion)
          mensaje <- paste0("-m 'ff=", v_ff, " md=", v_maxdepth, " ms=", v_minsplit, " mb=", v_minbucket, " cp=", v_cp, " trees=32'")
          linea <- paste(comando, competencia, arch, mensaje)
          salida <- system(linea, intern = TRUE)
          cat(salida, "\n")

          # Registrar corrida en tabla de resumen
          tb_resumen_corridas <- rbind(
            tb_resumen_corridas,
            list(
              iter = iter,
              feature_fraction = v_ff,
              maxdepth = v_maxdepth,
              minsplit = v_minsplit,
              minbucket_fraction = v_minbucket_frac,
              minbucket = v_minbucket,
              cp = v_cp,
              num_trees = PARAM$num_trees_max,
              archivo = archivo_prediccion,
              envios_positivos = cant_positivos
            )
          )

          # Guardar tabla de resumen actualizada en Google Drive
          fwrite(tb_resumen_corridas, file = "resumen_corridas.txt", sep = "\t")
        }
      }
    }
  }
}

In [ ]:
# Mostrar tabla de resumen de las corridas realizadas
tb_resumen_corridas

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")



---

